In [31]:
import heapq
import random
import time
ROWS, COLS = 10, 10
START = (0, 0)
GOAL = (9, 9)
def generate_grid(prob=0.2):
    grid = [[0 if random.random() > prob else 1 for _ in range(COLS)] for _ in range(ROWS)]
    grid[START[0]][START[1]] = 0
    grid[GOAL[0]][GOAL[1]] = 0
    return grid
def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])
def a_star(grid, start, goal):
    open_list = []
    heapq.heappush(open_list, (0, start))
    came_from = {}
    g_score = {start: 0}
    while open_list:
        _, current = heapq.heappop(open_list)
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            return path[::-1]
        for dx, dy in [(0,1),(1,0),(0,-1),(-1,0)]:
            nx, ny = current[0] + dx, current[1] + dy
            neighbor = (nx, ny)
            if 0 <= nx < ROWS and 0 <= ny < COLS and grid[nx][ny] == 0:
                tentative_g = g_score[current] + 1
                if neighbor not in g_score or tentative_g < g_score[neighbor]:
                    g_score[neighbor] = tentative_g
                    f = tentative_g + heuristic(neighbor, GOAL)
                    heapq.heappush(open_list, (f, neighbor))
                    came_from[neighbor] = current
    return None
def change_environment(grid, prob=0.1):
    for _ in range(5):
        x, y = random.randint(0, 9), random.randint(0, 9)
        if (x, y) not in [START, GOAL]:
            grid[x][y] = 1 if random.random() < prob else 0
def simulate():
    success = 0
    total_length = 0
    total_replan_time = 0
    for run in range(1, 51):
        grid = generate_grid()
        current = START
        path_length = 0
        replan_time = 0
        reached = False
        while current != GOAL:
            start_time = time.time()
            path = a_star(grid, current, GOAL)
            elapsed = time.time() - start_time
            replan_time += elapsed
            if not path:
                break
            current = path[0]
            path_length += 1
            change_environment(grid)
        if current == GOAL:
            success += 1
            reached = True
        total_length += path_length
        total_replan_time += replan_time
        print(f"Run {run}:")
        print("  Goal Reached:", reached)
        print("  Path Length:", path_length)
        print("  Replanning Time:", round(replan_time, 4), "seconds")
        print("-" * 40)
    print("\nFINAL SUMMARY (50 Runs)")
    print("Success Rate:", success / 50)
    print("Average Path Length:", total_length / 50)
    print("Average Replanning Time:", total_replan_time / 50)

simulate()

Run 1:
  Goal Reached: True
  Path Length: 18
  Replanning Time: 0.0056 seconds
----------------------------------------
Run 2:
  Goal Reached: True
  Path Length: 18
  Replanning Time: 0.0092 seconds
----------------------------------------
Run 3:
  Goal Reached: True
  Path Length: 18
  Replanning Time: 0.006 seconds
----------------------------------------
Run 4:
  Goal Reached: True
  Path Length: 24
  Replanning Time: 0.0057 seconds
----------------------------------------
Run 5:
  Goal Reached: True
  Path Length: 18
  Replanning Time: 0.0046 seconds
----------------------------------------
Run 6:
  Goal Reached: True
  Path Length: 18
  Replanning Time: 0.0038 seconds
----------------------------------------
Run 7:
  Goal Reached: True
  Path Length: 18
  Replanning Time: 0.0027 seconds
----------------------------------------
Run 8:
  Goal Reached: False
  Path Length: 16
  Replanning Time: 0.0025 seconds
----------------------------------------
Run 9:
  Goal Reached: True
  Pa

In [9]:
class ExpertSystem:
    def __init__(self):
        self.facts = set()
        self.rules = []
        self.trace = []
    def add_rule(self, conditions, conclusion):
        self.rules.append((conditions, conclusion))
    def add_fact(self, fact):
        self.facts.add(fact)
    def infer(self):
        inferred = True
        while inferred:
            inferred = False
            for conditions, conclusion in self.rules:
                if all(c in self.facts for c in conditions) and conclusion not in self.facts:
                    self.facts.add(conclusion)
                    self.trace.append(f"Inferred {conclusion} from {conditions}")
                    inferred = True
    def ask_user(self, fact):
        ans = input(f"Is '{fact}' true? (y/n): ").lower()
        if ans == 'y':
            self.add_fact(fact)
    def diagnose(self):
        for fact in ["high_traffic", "no_user_activity", "many_ports", "unknown_files"]:
            if fact not in self.facts:
                self.ask_user(fact)
        self.infer()
        for f in self.facts:
            if f.startswith("attack_"):
                return f
        return "No threat detected"
es = ExpertSystem()
es.add_rule(["high_traffic", "no_user_activity"], "attack_DDoS")
es.add_rule(["many_ports"], "attack_Port_Scan")
es.add_rule(["unknown_files"], "attack_Malware")
test_cases = [
    {"high_traffic", "no_user_activity"},
    {"many_ports"},
    {"unknown_files"},
    {"high_traffic"},
    {"no_user_activity"}
]
correct = 0
for i, case in enumerate(test_cases):
    es.facts.clear()
    es.trace.clear()
    for fact in case:
        es.add_fact(fact)
    result = es.diagnose()
    print(f"\nTest Case {i+1}: {case}")
    print("Diagnosis:", result)
    print("Reasoning Trace:")
    for t in es.trace:
        print("-", t)
    if result != "No threat detected":
        correct += 1
print("\nAccuracy:", correct / len(test_cases))


Test Case 1: {'no_user_activity', 'high_traffic'}
Diagnosis: attack_Port_Scan
Reasoning Trace:
- Inferred attack_DDoS from ['high_traffic', 'no_user_activity']
- Inferred attack_Port_Scan from ['many_ports']

Test Case 2: {'many_ports'}
Diagnosis: attack_Port_Scan
Reasoning Trace:
- Inferred attack_Port_Scan from ['many_ports']

Test Case 3: {'unknown_files'}
Diagnosis: attack_Port_Scan
Reasoning Trace:
- Inferred attack_DDoS from ['high_traffic', 'no_user_activity']
- Inferred attack_Port_Scan from ['many_ports']
- Inferred attack_Malware from ['unknown_files']

Test Case 4: {'high_traffic'}
Diagnosis: No threat detected
Reasoning Trace:

Test Case 5: {'no_user_activity'}
Diagnosis: attack_Malware
Reasoning Trace:
- Inferred attack_DDoS from ['high_traffic', 'no_user_activity']
- Inferred attack_Malware from ['unknown_files']

Accuracy: 0.8


In [ ]:
rules = [
    {
        "if": {"traffic": "high", "response_time": "slow"},
        "then": {"status": "congested"}
    },
    {
        "if": {"status": "congested", "source_type": "multiple_ips"},
        "then": {"diagnosis": "DDoS Attack"}
    },
    {
        "if": {"status": "congested", "source_type": "single_ip"},
        "then": {"diagnosis": "DoS Attack"}
    },
    {
        "if": {"connection_count": "high", "target_ports": "varied"},
        "then": {"diagnosis": "Port Scan"}
    }
]


def run_diagnosis(initial_facts):
    knowledge = initial_facts.copy()

    applied = True
    while applied:
        applied = False
        for rule in rules:
            if all(knowledge.get(k) == v for k, v in rule["if"].items()):
                for out_key, out_val in rule["then"].items():
                    if out_key not in knowledge:
                        knowledge[out_key] = out_val
                        applied = True

    return knowledge.get("diagnosis", "Unknown Threat")




scenarios = [
    {"traffic": "high", "response_time": "slow", "source_type": "multiple_ips"},
    {"traffic": "high", "response_time": "slow", "source_type": "single_ip"},
    {"connection_count": "high", "target_ports": "varied"},
    {"traffic": "normal"}
]


print(f"{'Scenario':<10} | {'Diagnosis':<20}")
print("-" * 35)

for i, logs in enumerate(scenarios, start=1):
    result = run_diagnosis(logs)
    print(f"{i:<10} | {result:<20}")


Scenario   | Diagnosis           
-----------------------------------
1          | DDoS Attack         
2          | DoS Attack          
3          | Port Scan           
4          | Unknown Threat      


In [47]:
import pandas as pd
import numpy as np
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

# -------------------------------------------------
# LOAD DATA (ROBUST FOR KAGGLE / STUDENT DATASETS)
# -------------------------------------------------
def load_data(file_path="student_data.csv"):
    print("Loading dataset:", file_path)
    
    # Try common separators
    for sep in [',', ';']:
        df = pd.read_csv(file_path, sep=sep, encoding="latin1", encoding_errors="ignore")
        # Check if 'G3' is present
        if 'G3' in df.columns:
            print(f"✅ Dataset loaded successfully with separator '{sep}'")
            return df
    # If not found, raise error
    raise ValueError("❌ Could not find 'G3' column. Check your CSV separator or file format.")


# -------------------------------------------------
# PREPROCESS DATA (AUTO-ADAPTIVE)
# -------------------------------------------------
def preprocess_data(df):
    # Strip whitespace from column names
    df.columns = df.columns.str.strip()
    
    # ----- TARGET CREATION -----
    if 'G3' in df.columns:
        df['passed'] = np.where(df['G3'] >= 10, 1, 0)
        df.drop([c for c in ['G1', 'G2', 'G3'] if c in df.columns],
                axis=1, inplace=True)
    else:
        raise ValueError(
            "❌ Target column not found.\n"
            "Expected final grade column named 'G3'.\n"
            "Your columns are:\n"
            f"{list(df.columns)}"
        )

    X = df.drop('passed', axis=1)
    y = df['passed']

    # ----- ENCODE CATEGORICAL FEATURES -----
    categorical_cols = X.select_dtypes(include=['object']).columns
    if len(categorical_cols) > 0:
        X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

    # ----- HANDLE MISSING VALUES -----
    imputer = SimpleImputer(strategy='mean')
    X_imputed = imputer.fit_transform(X)

    # ----- SCALE FEATURES -----
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imputed)

    X_final = pd.DataFrame(X_scaled, columns=X.columns)

    return X_final, y


# -------------------------------------------------
# FEATURE SELECTION
# -------------------------------------------------
def select_features(X, y, top_n=15):
    print("\nSelecting important features using Random Forest...")

    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X, y)

    importances = pd.Series(rf.feature_importances_, index=X.columns)
    top_features = importances.nlargest(top_n).index

    print("Top 5 Selected Features:")
    for f in top_features[:5]:
        print(" -", f)

    return X[top_features]


# -------------------------------------------------
# TRAIN MODELS & EVALUATE
# -------------------------------------------------
def run_experiment():
    df = load_data()
    X, y = preprocess_data(df)
    X_selected = select_features(X, y)

    X_train, X_test, y_train, y_test = train_test_split(
        X_selected, y,
        test_size=0.30,
        random_state=42
    )

    models = {
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Naive Bayes": GaussianNB(),
        "SVM": SVC(probability=True, random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "MLP (Deep Learning)": MLPClassifier(
            hidden_layer_sizes=(50, 30),
            max_iter=500,
            random_state=42
        )
    }

    # ----- HYBRID ENSEMBLE -----
    hybrid_estimators = [
        ('rf', models["Random Forest"]),
        ('mlp', models["MLP (Deep Learning)"]),
        ('svm', models["SVM"])
    ]

    models["Hybrid Ensemble"] = VotingClassifier(
        estimators=hybrid_estimators,
        voting='soft'
    )

    print(f"\n{'Model':<25} | Accuracy")
    print("-" * 40)

    results = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        results[name] = acc
        print(f"{name:<25} | {acc:.4f}")

    return results


# -------------------------------------------------
# MAIN
# -------------------------------------------------
if __name__ == "__main__":
    print("Starting experiment...")
    run_experiment()


Starting experiment...
Loading dataset: student_data.csv
✅ Dataset loaded successfully with separator ','

Selecting important features using Random Forest...
Top 5 Selected Features:
 - absences
 - failures
 - goout
 - age
 - health

Model                     | Accuracy
----------------------------------------
Decision Tree             | 0.6891
Naive Bayes               | 0.6975
SVM                       | 0.6891
Random Forest             | 0.6891
MLP (Deep Learning)       | 0.6303
Hybrid Ensemble           | 0.6723
